# DAS Perimeter Benchmark: explore the dataset
Five classes, 4,000 samples, four aligned representations.

This CPU tutorial explores the feature CSV and trains the RBF-SVM using the fixed benchmark protocol.
The **optional** full-data step downloads about 1.55 GB (allow 3.2 GB disk space).
SVM training needs no large array download or GPU. Your run is saved separately from the reference results.

**License:** tutorial code is MIT; data, weights and documentation are CC BY-NC 4.0.
See the [repository](https://github.com/StepsEight/DAS-Perimeter-Benchmark).
Run cells in order. Colab storage is temporary; downloaded files may disappear after the runtime ends.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path(os.environ.get("DAS_REPO_DIR", str(Path.cwd() / "DAS-Perimeter-Benchmark"))).resolve()
if not (REPO / "data/metadata.csv").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/StepsEight/DAS-Perimeter-Benchmark.git", str(REPO)], check=True)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)


## 1. Load the sample index
Colab normally includes the reader libraries. Install missing libraries if needed;
this exploration uses the runtime's available versions. For the paper's pinned
training environment, follow `requirements.txt` and `docs/benchmark.md` locally.


In [ ]:
import importlib.util
packages = {"numpy": "numpy", "pandas": "pandas", "scipy": "scipy",
            "h5py": "h5py", "yaml": "PyYAML", "matplotlib": "matplotlib", "sklearn": "scikit-learn", "joblib": "joblib"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import pandas as pd
from IPython.display import display
from src.dataset import load_metadata, load_representation

metadata = load_metadata()  # Checks hashes and recording-group separation.
display(metadata.groupby(["class_name", "split"]).size().unstack()[["train", "val", "test"]])
print("Samples:", len(metadata), "| Source recordings:", metadata.group_id.nunique())
display(metadata[["sample_id", "class_name", "source_bin_index", "group_id", "split"]].head(8))


Four consecutive samples per class come from one source recording at different
spatial locations. Preserve the supplied chronological train/validation/test splits;
do not randomly split these individual samples.

## 2. Inspect one sample and its 15 handcrafted features
Change `SAMPLE_ID` to another entry in the metadata. `vehicle` means bicycle.
This step does **not** need the large dataset download.


In [ ]:
SAMPLE_ID = "walking_000241"
rows = metadata.loc[metadata.sample_id.eq(SAMPLE_ID)]
if len(rows) != 1:
    raise ValueError("Choose a sample_id from metadata")
row = rows.iloc[0]
display(rows[["sample_id", "class_name", "split", "group_id", "channel_start", "channel_end"]])
features = pd.read_csv(REPO / "data/handcrafted_features.csv")
display(features.loc[features.sample_id.eq(SAMPLE_ID)].T)


## 3. Train and select the SVM
The included `data/handcrafted_features.csv` contains **15 features for every sample**.
We use the same implementation as the local benchmark: fit `StandardScaler` on
the **2,400 training samples only**, compare the 20 configured RBF-SVM settings
on the **800 validation samples**, and select the highest validation Macro-F1.
The selected training-only fit is then evaluated on the **800 test samples**.
Do not tune parameters using the test results.

This CPU step requires neither PyTorch nor MAT/NPZ files. It saves a new run under
`runs/colab_svm/svm/`; rerunning a completed configuration displays its saved results.
The tutorial uses available runtime libraries; the pinned local requirements are
provided for exact numerical comparisons.


In [ ]:
import yaml
from src.utils import load_config

run_config = load_config()
run_config["paths"]["output_root"] = "runs/colab_svm"
config_path = REPO / "runs/colab_svm/config.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(yaml.safe_dump(run_config, sort_keys=False), encoding="utf-8")
subprocess.run([sys.executable, str(REPO / "scripts/train.py"), "--model", "svm",
                "--config", str(config_path)], cwd=REPO, check=True)


## 4. Inspect and save your results
The table below reports your validation and final test scores in percent.
The saved checkpoint contains both the training-fitted scaler and selected SVM;
keep them together when applying the model to new feature vectors.


In [ ]:
import json
from IPython.display import Image

RUN_DIR = REPO / "runs/colab_svm/svm"
summary = json.loads((RUN_DIR / "run_summary.json").read_text(encoding="utf-8"))
print("Selected parameters:", summary["best_parameters"])
keys = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
scores = pd.DataFrame({split: {key: 100 * summary[split][key] for key in keys}
                       for split in ["validation", "test"]}).T
scores.columns = ["Accuracy (%)", "Macro-P (%)", "Macro-R (%)", "Macro-F1 (%)"]
display(scores.round(3))
display(Image(filename=str(RUN_DIR / "test_confusion_matrix.png")))
print("Model and scaler:", RUN_DIR / "best_pipeline.joblib")
print("Predictions:", RUN_DIR / "test_predictions.csv")


In [ ]:
import shutil
from IPython.display import FileLink

# Save the tutorial output with the noncommercial data/weights license.
shutil.copyfile(REPO / "LICENSE-DATA", RUN_DIR / "LICENSE")
(RUN_DIR / "README.txt").write_text(
    "DAS Perimeter Benchmark SVM tutorial outputs.\n"
    "Trained weights and results: CC BY-NC 4.0; see LICENSE.\n"
    "Attribution: Yinghuan Li, Changyuan Yu, Alan Pak Tao Lau.\n"
    "https://github.com/StepsEight/DAS-Perimeter-Benchmark\n"
    "These files describe this tutorial run; reference results are in the repository.\n",
    encoding="utf-8")
RESULTS_ZIP = Path(shutil.make_archive(str(REPO / "runs/colab_svm_results"), "zip", RUN_DIR))
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULTS_ZIP)))
else:
    files.download(str(RESULTS_ZIP))


## 5. Optional: download the full dataset
Set the flag to `True` to download and verify the approximately 1.55 GB archive.
This provides all five 2D MAT files, the combined 1D MAT, and STFT maps.
Already installed arrays are reused. The next cell plots the selected sample.


In [ ]:
DOWNLOAD_FULL_DATA = False
required = [REPO / "data/temporal/DAS_1D_2kHz_4000.mat",
            REPO / "data/time_frequency/stft.npz",
            *[REPO / f"data/spatiotemporal/{name}.mat" for name in metadata.class_name.unique()]]
if DOWNLOAD_FULL_DATA and not all(path.is_file() for path in required):
    subprocess.run([sys.executable, str(REPO / "scripts/download_data.py")], cwd=REPO, check=True)
DATA_READY = all(path.is_file() for path in required)
print("Arrays available." if DATA_READY else "Optional download skipped. Set DOWNLOAD_FULL_DATA = True to enable the plots.")


In [ ]:
if DATA_READY:
    import numpy as np
    import matplotlib.pyplot as plt
    from src.mat_loader import inspect_mat, read_samples
    from src.preprocessing import standardize_patch
    from src.dataset import verify_paths

    _, waveforms = load_representation("cnn1d")
    _, maps = load_representation("stft_cnn")
    verify_paths([row.spatiotemporal_file])
    source = inspect_mat(REPO / row.spatiotemporal_file)
    raw, valid, warnings = read_samples(source, [int(row.instance_index) - 1], restore_amplitude=True)
    if warnings:
        raise ValueError(warnings)
    patch = standardize_patch(raw[0], valid)
    position = int(row.row_index)
    time = np.arange(2000) / 2000
    fig, axes = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)
    im = axes[0].pcolormesh(time, np.arange(1, 51), patch, shading="auto", cmap="RdBu_r")
    axes[0].set(ylabel="Relative channel", title=f"{SAMPLE_ID} | {row.split} | Space-time")
    fig.colorbar(im, ax=axes[0], label="Standardized amplitude")
    axes[1].plot(time, waveforms[position], linewidth=0.8)
    axes[1].set(ylabel="Standardized amplitude", title="Fixed 25th-channel waveform")
    im = axes[2].pcolormesh(np.arange(33) * 64 / 2000, np.arange(129) * 2000 / 256,
                            maps[position], shading="auto", cmap="viridis")
    axes[2].set(xlabel="Time (s)", ylabel="Frequency (Hz)", title="Standardized log-power STFT")
    fig.colorbar(im, ax=axes[2], label="Standardized log power")
    for ax in axes:
        ax.set_xlim(0, 1)
    plt.show()
else:
    print("Run the optional download cell with DOWNLOAD_FULL_DATA = True first.")


The first 41 waveform points are padding, not measurements. The 2D view uses
one mean/std over valid time and space; the 1D waveform uses valid-time statistics;
the STFT map uses its own map-wide standardization. Values are not physical units.

## 6. Continue with the CNN benchmarks
The four baselines are RBF-SVM, 1D-CNN, 2D-CNN and STFT-CNN.
Checkpoints were selected by **validation Macro-F1**. The test set is reserved for
final evaluation, not model selection.

- [Dataset specification](https://github.com/StepsEight/DAS-Perimeter-Benchmark/blob/main/docs/dataset.md)
- [Training and evaluation protocol](https://github.com/StepsEight/DAS-Perimeter-Benchmark/blob/main/docs/benchmark.md)
- [Saved results](https://github.com/StepsEight/DAS-Perimeter-Benchmark/blob/main/results/summary.csv)
- [Checkpoint guide](https://github.com/StepsEight/DAS-Perimeter-Benchmark/blob/main/checkpoints/README.md)

For reproducible local runs, install the pinned requirements in the repository.
Use `python scripts/train.py --model all` to train or
`python scripts/evaluate.py --model stft_cnn --split test` to evaluate released weights.
Please cite `CITATION.cff` and retain the noncommercial data license when sharing.
